# Derm ReasonEdit — Edit & Eval (given user, given VLM)

Runs the **top-1 reliability** experiment end-to-end for one `(user, model)` pair:

1. **Build edit set** = the model's top-1 *errors* (so baseline top-1 acc should be ~0).
2. **Baseline** — ask the single-diagnosis question, record top-1.
3. **Edit** — add the human's corrected reasoning to the ReasonEdit codebook (`dual_sbert`).
4. **Apply retrieval + re-ask** — record post-edit top-1 (should flip toward the human answer).
5. **Report** reliability before vs after + qualitative cards, and save a results JSON.

This is the notebook twin of `run_efficacy.py` — it imports the same helpers so results match the CLI/sbatch path.

> **Env:** use the existing **`dermato_llama`** kernel — it already runs both models (transformers 4.57.1,
> peft 0.16.0) and has every revlm dep (sentence-transformers, scipy, matplotlib, PIL). See `../env/README.md`.
> Both models are already cached on scratch (`XDG_CACHE_HOME`), so no download. Set `HF_TOKEN` for gated repos.

## 0. Switches — pick the user and the VLM

In [1]:
import sys
from pathlib import Path

# Locate the reliability package regardless of the kernel's cwd (JupyterLab often
# starts the kernel at the server root, not the notebook folder).
def _find_pkg():
    hard = Path("/scratch/jq2uw/MME/derm_vlm_edit/reliability")
    for p in [Path.cwd(), Path.cwd().parent, hard, *Path.cwd().parents]:
        if (p / "config.py").exists() and p.name == "reliability":
            return p
        cand = p / "derm_vlm_edit" / "reliability"
        if (cand / "config.py").exists():
            return cand
    return hard

PKG = _find_pkg()
assert (PKG / "config.py").exists(), f"reliability/config.py not found (looked near {PKG})"
sys.path.insert(0, str(PKG))
print("package root:", PKG)

import config as C
sys.path.insert(0, str(C.DERM_EDIT_ROOT))

# ---- experiment switches -------------------------------------------------
USER           = "lhunterellul"   # "jesse.y.liu.md@gmail.com" | "achiou@stanford.edu" | "lhunterellul"
MODEL_SHORT    = "dermatollama"          # "medgemma" | "dermatollama"
RATIONALE_MODE = C.RATIONALE_MODE    # "edited_only" (human-authored) | "final"
SUBSAMPLE      = 0                     # 0 = use all edits; else cap for a quick smoke run
MAX_NEW_TOKENS = 32

ANNOT_MODEL = C.MODEL_SHORT_TO_ANNOT[MODEL_SHORT]
print(f"user   = {USER}")
print(f"model  = {MODEL_SHORT}  (annotation label: {ANNOT_MODEL})")
print(f"rationale_mode = {RATIONALE_MODE} | subsample = {SUBSAMPLE} | max_new_tokens = {MAX_NEW_TOKENS}")
print(f"annotations exists: {C.ANNOTATIONS_CSV.exists()} | images exists: {C.IMAGE_ROOT.exists()}")

package root: /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/reliability
user   = lhunterellul
model  = dermatollama  (annotation label: DermatoLlama)
rationale_mode = edited_only | subsample = 0 | max_new_tokens = 32
annotations exists: True | images exists: True


## 1. Build the edit set (model's top-1 errors for this user)

In [2]:
import json
from data_engineering.build_edit_set import build as build_edit_set

edits, summary = build_edit_set(USER, ANNOT_MODEL, RATIONALE_MODE, write=True)
if SUBSAMPLE and len(edits) > SUBSAMPLE:
    edits = edits[:SUBSAMPLE]

assert edits, f"No usable edits for user={USER} model={ANNOT_MODEL}."
print(json.dumps({k: v for k, v in summary.items() if not k.endswith("_counts")}, indent=2))
print(f"\n{len(edits)} edits -> {C.edit_set_path(USER, ANNOT_MODEL)}")

{
  "user": "lhunterellul",
  "model": "DermatoLlama",
  "rationale_mode": "edited_only",
  "question": "What is the single most likely diagnosis for this lesion?",
  "n_edits": 12,
  "n_images_missing": 0,
  "n_multi_dx_top1": 6,
  "n_y16_unmapped": 0,
  "n_empty_rationale": 0
}

12 edits -> /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/reliability/results/edit_set__lhunterellul__DermatoLlama.jsonl


## 2. Load the VLM + resolve ReasonEdit layer names
First model load downloads weights (can be slow / needs `HF_TOKEN` for gated repos).

In [3]:
import torch

sys.path.insert(0, str(C.DERM_EDIT_ROOT))
from reasonedit import ReasonEdit, VQAModel, load_config

# reuse the exact helpers from the CLI runner so notebook == run_efficacy.py
from run_efficacy import is_correct, resolve_layers, cap_n_pre
from derm_dataset import DermEditDataset

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# Editor defaults (in ReasonEdit.__init__) already match rankedit answer-only IKE:
# key_type=answer, merge_keys=False, hubness_keys=False, cap_keys=1, n_radius_samples=0.
config = load_config(
    MODEL_SHORT, device=device,
    batch_size=1, max_new_tokens=MAX_NEW_TOKENS, seed=42,
)

model = VQAModel(config)
resolve_layers(model, config)   # corrects best-guess inner_params_* against the loaded tree
print("device:", device)
print("editor defaults: key_type/merge/hubness/cap_keys live in ReasonEdit.__init__")

/home/jq2uw/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|████████████████████████████████████████████████| 5/5 [00:02<00:00,  1.83it/s]


  Suffix-matched layer: base_model.model.model.vision_model.transformer.layers.28.mlp.fc1.weight
[layers] inner_params_vision: ['base_model.model.model.vision_model.transformer.layers.28.mlp.fc1.weight']
  Found corrected layer: base_model.model.model.language_model.layers.0.mlp.gate_proj.base_layer.weight
[layers] inner_params_lang: ['base_model.model.model.language_model.layers.0.mlp.gate_proj.base_layer.weight']
  Found corrected layer: base_model.model.model.language_model.layers.0.mlp.gate_proj.base_layer.weight
[layers] inner_params: ['base_model.model.model.language_model.layers.0.mlp.gate_proj.base_layer.weight']
device: cuda:0
editor defaults: key_type/merge/hubness/cap_keys live in ReasonEdit.__init__


### 2.5 Sanity — confirm the ReasonEdit vision hook layer resolves (GPU)
`dual_sbert` only hooks `inner_params_vision[0]`. `resolve_layers` (cell above) already mapped the
prefix-free yaml name to the exact live param (Mllama is a PeftModel, so names carry a
`base_model.model.` prefix). This cell verifies that resolution against the loaded model.

In [4]:
import re
from reasonedit.utils import validate_and_correct_param_name

params = dict(model.model.named_parameters())
resolved = config.model.inner_params_vision[0]   # already corrected by resolve_layers
print("inner_params_vision ->", resolved)
print("exists in model     :", resolved in params,
      "| shape:", tuple(params[resolved].shape) if resolved in params else None)

# show the tail of the vision-MLP weight list so you can pick a different late block if desired
vis = sorted(n for n in params if re.search(r"vision.*mlp\.(fc1|fc2|up_proj|down_proj)\.weight$", n))
print(f"\n{len(vis)} vision-mlp weight params; last 5:")
for n in vis[-5:]:
    print("  ", n)

inner_params_vision -> base_model.model.model.vision_model.transformer.layers.28.mlp.fc1.weight
exists in model     : True | shape: (5120, 1280)

80 vision-mlp weight params; last 5:
   base_model.model.model.vision_model.transformer.layers.7.mlp.fc2.weight
   base_model.model.model.vision_model.transformer.layers.8.mlp.fc1.weight
   base_model.model.model.vision_model.transformer.layers.8.mlp.fc2.weight
   base_model.model.model.vision_model.transformer.layers.9.mlp.fc1.weight
   base_model.model.model.vision_model.transformer.layers.9.mlp.fc2.weight


## 3. Baseline top-1 (pre-edit)

In [5]:
ds = DermEditDataset(config, edits)
ds.set_dataloader()

print("generating pre-edit top-1 ...")
ds.task_generate(model, use_cache=True, max_new_tokens=MAX_NEW_TOKENS)
pre = [ex.get("pred", {}).get("answer", "") for ex in ds.data]
for ex, pa in list(zip(ds.data, pre))[:5]:
    print(f"  {ex['case_id']:>16}  pred={pa!r:40}  human={ex['answer']!r}")

generating pre-edit top-1 ...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


     1002_combined  pred='Basal cell carcinoma'                    human='squamous cell carcinoma in situ'
     1015_combined  pred='Nevus'                                   human='blue nevus'
     1019_combined  pred='Nevus'                                   human='blue nevus'
     1025_combined  pred='Basal cell carcinoma'                    human='actinic keratosis'
      109_combined  pred='Nevus'                                   human='dysplastic nevus'


In [6]:
import torch
from PIL import Image

ex = ds.data[0]
img = ex["image"]
img = Image.open(img).convert("RGB") if isinstance(img, str) else img

inputs = model.preprocess([img], [ex["prompt"]], model.processor)   # CPU tensors
ids = inputs["input_ids"]
emb = model.model.get_input_embeddings()
cfg = model.model.config
img_tok = getattr(cfg, "image_token_index", None) or getattr(cfg, "image_token_id", None)

print("keys        :", list(inputs.keys()))
print("input_ids   :", tuple(ids.shape), "min", int(ids.min()), "max", int(ids.max()))
print("num_embeds  :", emb.num_embeddings)
print("TOKEN OOB?  :", int(ids.max()) >= emb.num_embeddings)      # <-- the smoking gun
if "pixel_values" in inputs:
    print("pixel_values:", tuple(inputs["pixel_values"].shape))
if img_tok is not None:
    print("image_tok id:", img_tok, "count in ids:", int((ids == img_tok).sum()))

keys        : ['input_ids', 'attention_mask', 'pixel_values', 'aspect_ratio_ids', 'aspect_ratio_mask', 'cross_attention_mask']
input_ids   : (1, 31) min 13 max 128256
num_embeds  : 128264
TOKEN OOB?  : False
pixel_values: (1, 1, 4, 3, 560, 560)
image_tok id: 128256 count in ids: 1


## 4. Edit (build codebook) → apply retrieval → regenerate

In [7]:
print("building ReasonEdit codebook ...")
editor = ReasonEdit(config, model)
editor.edit(config, edit_ds=ds)
editor.apply_to_dataset(ds)

print("generating post-edit top-1 ...")
ds.task_generate(model, use_cache=True, max_new_tokens=MAX_NEW_TOKENS)
post = [ex.get("pred", {}).get("answer", "") for ex in ds.data]

retr_log = {r["uid"]: r.get("facts", []) for r in (editor.last_retrieval_log or [])}
for ex, qa in list(zip(ds.data, post))[:5]:
    print(f"  {ex['case_id']:>16}  post={qa!r:40}  human={ex['answer']!r}")

building ReasonEdit codebook ...
[ReasonEdit] mode=dual_sbert, key_type=answer, pair_rationale_w=orig, merge=False, hubness=False, cap_keys=1
[ReasonEdit] edit 1/12...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 2/12...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 3/12...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 4/12...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 5/12...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 6/12...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 7/12...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 8/12...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 9/12...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 10/12...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 11/12...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 12/12...[Keys] +1 added, 0 merged (from 1 candidates)
[Re

## 5. Reliability before vs after (the 0 → 1 flip)

In [8]:
rows = []
n_pre = n_post = 0
for ex, pa, qa in zip(ds.data, pre, post):
    hn, hy = ex["answer"], ex.get("human_top1_y16")
    pc, qc = is_correct(pa, hn, hy), is_correct(qa, hn, hy)
    n_pre += int(pc); n_post += int(qc)
    rows.append({
        "uid": ex["uid"], "case_id": ex.get("case_id"),
        "model_top1": ex.get("model_top1"),
        "human_top1": ex.get("human_top1_raw"), "human_norm": hn, "human_y16": hy,
        "pred_pre": pa, "pred_post": qa,
        "correct_pre": pc, "correct_post": qc,
        "retrieved_facts": retr_log.get(ex["uid"], []),
    })

n = len(rows)
n_pre = cap_n_pre(n_pre)
result = {
    "user": USER, "model": ANNOT_MODEL, "short_model": MODEL_SHORT,
    "rationale_mode": RATIONALE_MODE, "question": C.QUESTION,
    "n_edits": n,
    "reliability_pre": n_pre / n if n else 0.0,
    "reliability_post": n_post / n if n else 0.0,
    "n_correct_pre": n_pre, "n_correct_post": n_post,
    "inner_params_vision": getattr(config.model, "inner_params_vision", None),
    "cases": rows,
}
print(f"reliability_pre  = {result['reliability_pre']:.3f} ({n_pre}/{n})")
print(f"reliability_post = {result['reliability_post']:.3f} ({n_post}/{n})")
print(f"flipped wrong->right: {sum(r['correct_post'] and not r['correct_pre'] for r in rows)}")

reliability_pre  = 0.083 (1/12)
reliability_post = 1.000 (12/12)
flipped wrong->right: 4


## 6. Per-case table + before/after cards

In [9]:
for r in rows:
    flag = "OK " if r["correct_post"] else "-- "
    chg  = "  (flip)" if r["correct_post"] and not r["correct_pre"] else ""
    print(f"{flag}{r['case_id']:>16} | model:{str(r['model_top1'])[:22]:22} | "
          f"pre:{str(r['pred_pre'])[:22]:22} -> post:{str(r['pred_post'])[:22]:22} | "
          f"human:{r['human_top1']}{chg}")

OK    1002_combined | model:Basal Cell Carcinoma   | pre:Basal cell carcinoma   -> post:Squamous cell carcinom | human:SCCis, pigmented actinic keratosis  (flip)
OK    1015_combined | model:Seborrheic Keratosis   | pre:Nevus                  -> post:Blue nevus             | human:Blue nevus vs melanoma vs recurrent nevus
OK    1019_combined | model:Basal Cell Carcinoma   | pre:Nevus                  -> post:Blue nevus             | human:Blue nevus vs dysplastic nevus vs melanoma Vs pigmented basal cell carcinoma
OK    1025_combined | model:Basal Cell Carcinoma   | pre:Basal cell carcinoma   -> post:Actinic Keratosis      | human:AK/HAK, SCC in situ, SCC, Benign Lichenoid Keratosis  (flip)
OK     109_combined | model:Nevus                  | pre:Nevus                  -> post:Dysplastic Nevus       | human:Dysplastic nevus
OK    1134_combined | model:Nevus                  | pre:Nevus                  -> post:Dysplastic Nevus       | human:Dysplastic nevus
OK    1142_combined | model:N

In [10]:
# import textwrap
# import matplotlib.pyplot as plt
# from PIL import Image

# def show_cards(edits, rows, n=6):
#     n = min(n, len(edits))
#     fig, axes = plt.subplots(n, 2, figsize=(14, 3.3 * n), gridspec_kw={"width_ratios": [1, 2.6]})
#     if n == 1:
#         axes = axes.reshape(1, 2)
#     for i, (e, r) in enumerate(zip(edits[:n], rows[:n])):
#         try:
#             axes[i, 0].imshow(Image.open(e["image"]).convert("RGB"))
#         except Exception as exc:
#             axes[i, 0].text(0.5, 0.5, f"no image\n{exc}", ha="center")
#         axes[i, 0].axis("off")
#         axes[i, 0].set_title(("OK " if r["correct_post"] else "-- ") + str(e["case_id"]), fontsize=9)
#         facts = "\n".join(f"  - {f}" for f in r["retrieved_facts"]) or "  (none)"
#         txt = (
#             f"MODEL top-1 (wrong): {r['model_top1']}\n"
#             f"PRE  pred          : {r['pred_pre']}\n"
#             f"POST pred          : {r['pred_post']}\n"
#             f"HUMAN top-1        : {r['human_top1']}  (y16: {r['human_y16']})\n\n"
#             f"retrieved facts:\n{facts}\n\n"
#             f"human reasoning:\n{textwrap.fill(e.get('rationale', ''), 92)}"
#         )
#         axes[i, 1].axis("off")
#         axes[i, 1].text(0.0, 1.0, txt, va="top", ha="left", fontsize=8.5, family="monospace")
#     plt.tight_layout(); plt.show()

# show_cards(edits, rows, n=6)

## 7. Save results JSON (same schema as `run_efficacy.py`)

In [11]:
import os

os.makedirs(C.RESULTS_DIR, exist_ok=True)
safe_user = USER.replace("@", "_at_").replace(".", "_")
out = C.RESULTS_DIR / f"reliability__{safe_user}__{MODEL_SHORT}.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
print("saved ->", out)

saved -> /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/reliability/results/reliability__lhunterellul__dermatollama.json


## 8. Collect ALL runs → PI-ready before/after table
Run this **after** you've executed the notebook for each `(user, vlm)` combo (each writes its own
`results/reliability__*.json`). This cell scans them all and builds the 3×2 table
(Markdown + LaTeX + CSV in `results/`). It does not need the model loaded.

In [12]:
import importlib
import make_report as MR
importlib.reload(MR)
from IPython.display import Markdown, display

results = MR.load_results(C.RESULTS_DIR)
found = sorted(f"{u}/{sm}" for (u, sm) in results)
print(f"found {len(results)} runs:", found)

users = [u for u in MR.REPORT_USERS if any((u, sm) in results for sm, _ in MR.VLMS)] \
        or sorted({u for (u, _) in results})
rows, pooled = MR.build_rows(results, users, MR.VLMS)

md = MR.render_markdown(rows, pooled, MR.VLMS)
out = C.RESULTS_DIR / "report.csv"
MR.render_csv(rows, pooled, MR.VLMS, out)
print("wrote", out, "\n")

display(Markdown(md))

found 6 runs: ['achiou@stanford.edu/dermatollama', 'achiou@stanford.edu/medgemma', 'jesse.y.liu.md@gmail.com/dermatollama', 'jesse.y.liu.md@gmail.com/medgemma', 'lhunterellul/dermatollama', 'lhunterellul/medgemma']
wrote /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/reliability/results/report.csv 



| User | MedGemma Before | MedGemma After | MedGemma Δ | DermatoLlama Before | DermatoLlama After | DermatoLlama Δ |
|---|---|---|---|---|---|---|
| jesse.y.liu.md | 4% (1/25) | 100% (25/25) | +96 | 0% (0/15) | 100% (15/15) | +100 |
| achiou | 0% (0/21) | 100% (21/21) | +100 | 6% (1/17) | 100% (17/17) | +94 |
| lhunterellul | 0% (0/8) | 100% (8/8) | +100 | 8% (1/12) | 100% (12/12) | +92 |
| **Pooled** | 2% (1/54) | 100% (54/54) | +98 | 5% (2/44) | 100% (44/44) | +95 |
